In [1]:
import unicodedata
import json, copy, os

import pickle
from collections import Counter

import pandas as pd
from tqdm import tqdm

In [2]:
import tokenizer_edge_cases_char as ec

In [3]:
from tokenizers import Tokenizer, decoders, pre_tokenizers
from transformers import AutoTokenizer, PreTrainedTokenizerFast
from datasets import load_dataset

#### Loading the tokenizer and modifying its regex pattern

In [4]:
model_dir = "/Users/nirajh/Code/Language_adaptation/olmo2-7b-stage1"

# Loading the tokenizer
tok = AutoTokenizer.from_pretrained(model_dir)

## Changing the regex pattern in the pre tokenizer, so that kannada words are split correctly
tj = json.loads(tok.backend_tokenizer.to_str())        # Getting the tokenizer's vocab, merges, special tokens, and pre-processing configs
split = tj["pre_tokenizer"]["pretokenizers"][0]        # Fetches the first pre-tokenizer (they are usually chained together under a Sequence)
old = split["pattern"]["Regex"]
new = old.replace(r"[^\r\n\p{L}\p{N}]?\p{L}+", r"[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+")
assert new != old, "pattern not found; print(old) and adjust"
split["pattern"]["Regex"] = new

# Checking its output
kn_tok = Tokenizer.from_str(json.dumps(tj))
text = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
print([text[s:e] for _, (s, e) in kn_tok.pre_tokenizer.pre_tokenize_str(text)])

['ನಮಸ್ಕಾರ', ' ಕನ್ನಡ', ' ಭಾಷೆಯಲ್ಲಿ']


#### Loading the Kannada dataset

In [5]:
dataset_dir = "/Users/nirajh/Code/Language_adaptation/sangraha/train_split"
print(f"Loading the kannada dataset from the following path:\n{dataset_dir}\n")

# Load the dataset 
kn_train = load_dataset(
    "ai4bharat/sangraha",
    data_dir="verified/kan",
    cache_dir=dataset_dir,
    download_mode="reuse_cache_if_exists"
)

Loading the kannada dataset from the following path:
/Users/nirajh/Code/Language_adaptation/sangraha/train_split



Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/3632345 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/21 [00:00<?, ?it/s]

#### Loading the evaluation set

In [6]:
kn_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/kn_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        kn_texts.append(data["text"])

print(f"Successfully read {len(kn_texts)} lines.")   # Should be 1997


en_texts = []

with open("/Users/nirajh/Code/Language_adaptation/eval/en_eval_texts.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line)
        en_texts.append(data["text"])

print(f"Successfully read {len(en_texts)} lines.")   # Should be 2000 lines

Successfully read 1997 lines.
Successfully read 2000 lines.


#### Loading the new merges list

In [7]:
# Loading the new merges list
final_path = "/Users/nirajh/Code/Language_adaptation/merges_files/new_merges_excl.pkl"

with open(final_path, "rb") as f:
    merges_ = pickle.load(f)

In [8]:
merges_[:5]

[(('à³', 'į'), 574565151),
 (('à²', '¿'), 471364707),
 (('à²', '°'), 338308884),
 (('à³', 'ģ'), 323417073),
 (('à²', '¾'), 310022063)]

#### Loading the Unigram Tokens

In [9]:
# Loading the unigram tokenizer
u_tok = Tokenizer.from_file("/Users/nirajh/Code/Language_adaptation/tokenizers/unigram/unigram_4000_char.json")

# Getting the original vocabulary of the OLMo tokenizer
orig_vocab = tok.get_vocab()

In [10]:
# Fetching the tokens that are in the Unigram tokenizer's vocab, but are not present in OLMo's original tokenizer
u_new_tokens = ec.unigram_char_new_tokens(u_tok, orig_vocab)
print("Unigram new Kannada tokens:", len(u_new_tokens))

Unigram new Kannada tokens: 3998


These are the new Kannada tokens that the Unigram tokenizer has learnt. Unigram doesn't learn merges, only tokens (a token that it has learnt could only be a part of a word). The 3,508 tokens we see here will be a part of the model vocabulary (if this tokenizer is used).
<br><br>
<b>Why 3,508 and not 4,000? See below. </b>

In [11]:
# Checking some of these tokens
for t in list(u_new_tokens)[:5]:
    print(t)
    print(ec.to_text(t))
    print()

Ġà²ľà²²
 ಜಲ

Ġà²¤à³Ĩà²Ĺà³Ĩà²¦à³ģ
 ತೆಗೆದು

Ġà²¹à³ģà²Łà³įà²Łà³ģà²¹à²¬à³įà²¬
 ಹುಟ್ಟುಹಬ್ಬ

Ġà²¸à²¦à²¸à³įà²¯à²°
 ಸದಸ್ಯರ

à³Ī
ೈ



In [12]:
def category(p):
    if p == "<unk>": return "<unk>"
    bl = ec.to_bytelevel(p)
    if len(p.lstrip(" ")) <= 1: return "single character or space"
    if ec.is_kannada(bl): return "Kannada, already in OLMo" if bl in orig_vocab else "Kannada, new"
    return "other"

print(*Counter(category(p) for p in u_tok.get_vocab()).items(), sep="\n")

('Kannada, new', 3881)
('single character or space', 118)
('<unk>', 1)


#### Initializing function to build the BPE tokenizer

In [13]:
# Getting the JSON version of the regex-modded tokenizer
base_tj = json.loads(kn_tok.to_str())          # regex-modified, no new merges yet
base_tj["normalizer"] = {"type": "NFC"}        # same normalization the merges were learned on

In [14]:
# Initializing function to build a tokenizer with the new list of merges (path to merges list to be given)
def build_tokenizer(n, out_dir, new_merges, model_path):
    
    tkj = copy.deepcopy(base_tj)
    vocab = tkj["model"]["vocab"]
    merge_list = tkj["model"]["merges"]
    as_str = isinstance(merge_list[0], str)     # "a b" or ["a", "b"], depending on version
    next_id = max(max(vocab.values()), max(t["id"] for t in tkj["added_tokens"])) + 1

    for (a, b), _ in new_merges[:n]:
        if a + b not in vocab:                 # reuse the ID if the token already exists
            vocab[a + b] = next_id
            next_id += 1
        merge_list.append(f"{a} {b}" if as_str else [a, b])

    os.makedirs(out_dir, exist_ok=True)
    
    # return AutoTokenizer.from_pretrained(out_dir)
    # The issue with the above line: It doesn't just read tokenizer.json. It also reads tokenizer_config.json.
    # In recent transformers versions, model-specific tokenizer classes can rebuild parts of the pipeline, such as the pre-tokenizer and normalizer, from their own built-in defaults instead of using what’s in tokenizer.json
    # This means the tokenizer is not using the new regex pattern. It is using the pre-built one and fragmenting kannada words.

    hf_tok = PreTrainedTokenizerFast(
        tokenizer_object=Tokenizer.from_str(json.dumps(tkj)),
        eos_token="<|endoftext|>",
        pad_token="<|pad|>",
    )
    hf_tok.save_pretrained(out_dir)     # writes tokenizer.json + a config naming PreTrainedTokenizerFast

    # fail loudly if the pipeline didn't survive the save/load round trip
    reloaded = AutoTokenizer.from_pretrained(out_dir)
    test = "ನಮಸ್ಕಾರ ಕನ್ನಡ ಭಾಷೆಯಲ್ಲಿ"
    pieces = [test[s:e] for _, (s, e) in reloaded.backend_tokenizer.pre_tokenizer.pre_tokenize_str(test)]
    assert pieces == ["ನಮಸ್ಕಾರ", " ಕನ್ನಡ", " ಭಾಷೆಯಲ್ಲಿ"], pieces
    assert reloaded.backend_tokenizer.normalizer is not None, "NFC normalizer was dropped"
    return reloaded



In [15]:
# Building BPE tokenizer
print(f"Number of new merges being used to build the BPE tokenizer: {len(u_new_tokens)}")
bpe_tok = build_tokenizer(len(u_new_tokens), f"/Users/nirajh/Code/Language_adaptation/tokenizers/kn_excl_{len(u_new_tokens)}",
                          merges_, model_dir)

# Getting the new tokens in the BPE tokenizer that are not in the original OLMo model
b_new = ec.bpe_new_tokens(bpe_tok)

Number of new merges being used to build the BPE tokenizer: 3998


In BPE, each merge merge usually creates one new token (unless the result already exists in the vocabulary). Therefore, it is a fair comparison to match `n` Unigram tokens with `n` BPE merges. Since the Unigram tokenizer has learnt 3,508 Kannada tokens, we are using the same number of BPE merges.

In [16]:
# Getting the base pre tokenizer with the modified regex pattern
pre = kn_tok.pre_tokenizer
segs = {"BPE": ec.cached(ec.bpe_segmenter(bpe_tok)),
        "Unigram": ec.cached(ec.hybrid_char_segmenter(u_tok, kn_tok.model))}

In [17]:
segs

{'BPE': <function tokenizer_edge_cases_char.cached.<locals>.f(w)>,
 'Unigram': <function tokenizer_edge_cases_char.cached.<locals>.f(w)>}

In [18]:
# Reading the word freq file
with open("/Users/nirajh/Code/Language_adaptation/merges_files/word_freq.pkl", "rb") as f:
    train_freq = pickle.load(f)["word_freq"]

In [19]:
train_freq.most_common(5)

[('.', 60116249),
 (',', 41402116),
 ('Ġ', 21435058),
 ('.Ċ', 18937315),
 ('Ġà²Ī', 9952450)]

In [20]:
print(type(kn_texts))
print(type(kn_train))

<class 'list'>
<class 'datasets.dataset_dict.DatasetDict'>


In [21]:
sample = kn_train["train"].shuffle(seed=7).select(range(20_000))
usage_texts = kn_texts + list(sample["text"])
print(len(usage_texts))   # 21,997

21997


In [22]:
b_new_kn = {t for t in b_new if ec.is_kannada(t)}
print(f"BPE new tokens: {len(b_new)}, containing Kannada: {len(b_new_kn)}")
print([ec.to_text(t) for t in b_new - b_new_kn])      # inspect any exceptions

BPE new tokens: 3998, containing Kannada: 3998
[]


Sanity check before comparisons:

In [23]:
print([ec.to_text(p) for w in ec.pretokens("ಕನ್\u200cನಡ (ಮನೆ)", pre) for p in segs["Unigram"](w)])

['ಕ', 'ನ್', '\u200c', 'ನ', 'ಡ', ' (', 'ಮನೆ', ')']


### Comparisons

<b>1. Morphology: Suffix bounday rate</b>

In [24]:
# Morphology: boundary at the suffix start (higher = more morpheme-aligned)
morph_res = ec.suffix_boundary_rate(kn_texts, pre, segs)

print("Suffix boundary rate\n")
pd.DataFrame({
    'Suffix': morph_res['BPE'].keys(),
    'BPE': [round(v[0], 5) for v in morph_res['BPE'].values()],
    'Unigram': [round(v[0], 5) for v in morph_res['Unigram'].values()],
    'Count': [v[1] for v in morph_res['Unigram'].values()],
})

Suffix boundary rate



,Suffix,BPE,Unigram,Count
0,ಗಳು,0.90487,0.84962,5792
1,ALL,0.28570,0.30200,67218
2,ನ್ನು,0.06748,0.10208,15782
3,ಗೆ,0.23573,0.30941,16315
4,ಲ್ಲಿ,0.05834,0.06184,15718
5,ಕ್ಕೆ,0.75931,0.87810,4512
6,ಗಳ,0.85674,0.79527,4230
7,ತ್ತಾರೆ,0.39435,0.00097,1027
8,ತ್ತದೆ,0.09256,0.00061,3295
9,ಲ್ಲದೆ,0.61154,0.88846,260


<b>2. Fertility by frequency range</b>

In [25]:
# Fertility by how common each word was in training
fertility_freq_res = ec.fertility_by_frequency(kn_texts, pre, segs, train_freq)

print("Fertility by frequency range \n")

df_freq = pd.DataFrame.from_dict(fertility_freq_res, orient='index')
df_freq = df_freq.round(5)
df_freq.index.name = 'Frequency range'
display(df_freq)

Fertility by frequency range 



,word occurrences,unique words,BPE,Unigram
Frequency range,,,,
>=10k,295443,10538,1.53043,1.47913
1k-10k,80517,33382,2.81738,2.82857
100-1k,39019,31152,3.30813,3.36085
3-100,26158,23651,3.90710,4.07157
<3 or unseen,8661,8548,5.02759,5.37340


<b>3 + 4. Token utilization </b>

In [26]:
# Dead tokens and utilization (one tokenization pass per tokenizer)
counts = {"BPE": ec.kannada_token_counts(usage_texts, pre, segs["BPE"]),
          "Unigram": ec.kannada_token_counts(usage_texts, pre, segs["Unigram"])}

In [27]:
print("New token usage \n")

df_counts = pd.DataFrame({
    'BPE': dict(ec.new_token_usage(counts["BPE"], b_new)),
    'Unigram': dict(ec.new_token_usage(counts["Unigram"], u_new_tokens))
})

df_counts.index.name = 'Token usage'
display(df_counts.round(3))

New token usage 



,BPE,Unigram
Token usage,,
new tokens,3998.000,3998.00
never used,0.000,10.00
used < 1,0.000,10.00
used < 10,4.000,14.00
used < 100,71.000,22.00
share of Kannada tokens that are new,0.994,0.97


<b>Entropy</b>

In [28]:
df_entropy = pd.DataFrame({
    'BPE': dict(ec.utilization(counts["BPE"])),
    'Unigram': dict(ec.utilization(counts["Unigram"]))
})

display(df_entropy.round(4))

,BPE,Unigram
distinct tokens used,5068.0000,4799.0000
Shannon entropy (bits),10.8538,10.3669
Renyi entropy alpha=2.5 (bits),9.2934,7.7949
Renyi efficiency,0.7551,0.6374


<b>5. Robustness and segmentations </b>

In [29]:
# Robustness and side-by-side segmentations
ec.show_segmentations(ec.ROBUSTNESS_EXAMPLES, pre, segs)

ಮರ ಮರವನ್ನು ಮರದಲ್ಲಿ ಮರದಿಂದ ಮರಗಳು ಮರಗಳಲ್ಲಿ
  BPE      ( 12 tokens): ಮ | ರ |  ಮರ | ವನ್ನು |  ಮ | ರದಲ್ಲಿ |  ಮರ | ದಿಂದ |  ಮರ | ಗಳು |  ಮರ | ಗಳಲ್ಲಿ
  Unigram  ( 12 tokens): ಮ | ರ |  ಮರ | ವನ್ನು |  ಮರ | ದಲ್ಲಿ |  ಮರ | ದಿಂದ |  ಮರ | ಗಳು |  ಮರ | ಗಳಲ್ಲಿ

ಮನೆ ಮನೆಯನ್ನು ಮನೆಯಲ್ಲಿ ಮನೆಗೆ ಮನೆಗಳು
  BPE      (  8 tokens): ಮ | ನೆ |  ಮನೆ | ಯನ್ನು |  ಮನೆಯಲ್ಲಿ |  ಮನೆಗೆ |  ಮನೆ | ಗಳು
  Unigram  (  7 tokens): ಮನೆ |  ಮನೆ | ಯನ್ನು |  ಮನೆಯಲ್ಲಿ |  ಮನೆಗೆ |  ಮನೆ | ಗಳು

ನಾನು ನಾಳೆ office ಗೆ ಹೋಗುತ್ತೇನೆ
  BPE      (  6 tokens): ನಾನು |  ನಾಳೆ |  office |  ಗೆ |  ಹೋಗು | ತ್ತೇನೆ
  Unigram  (  7 tokens): ನಾನು |  ನಾಳೆ |  office |   | ಗೆ |  ಹೋಗ | ುತ್ತೇನೆ

ನನ್ನ phone number ೯೮೮೦೧೨೩೪೫೬
  BPE      ( 14 tokens): ನನ್ನ |  phone |  number |   | ೯ | ೮ | ೮ | ೦ | ೧ | ೨ | ೩ | ೪ | ೫ | ೬
  Unigram  ( 15 tokens): ನ | ನ್ನ |  phone |  number |   | ೯ | ೮ | ೮ | ೦ | ೧ | ೨ | ೩ | ೪ | ೫ | ೬

ಏನ್ ಮಾಡ್ತಿದೀಯ
  BPE      (  6 tokens): ಏ | ನ್ |  ಮಾಡ್ | ತಿ | ದೀ | ಯ
  Unigram  (  6 tokens): ಏ | ನ್ |  ಮಾಡ್ತ | ಿ | ದ | ೀಯ

ಕನ್‌ನಡ ಕನ್ನಡ
  BPE      (  6 tokens): ಕ | ನ್ |

In [30]:
robustness_res = ec.robustness_stats(ec.ROBUSTNESS_EXAMPLES, pre, segs)

df = pd.DataFrame(robustness_res)
df.set_index('text').round(3)

,BPE tokens/word,BPE fragment rate,Unigram tokens/word,Unigram fragment rate
text,,,,
ಮರ ಮರವನ್ನು ಮರದಲ್ಲಿ ಮರದಿಂದ ಮರಗಳು ಮರಗಳಲ್ಲಿ,2.0,0.0,2.00,0.0
ಮನೆ ಮನೆಯನ್ನು ಮನೆಯಲ್ಲಿ ಮನೆಗೆ ಮನೆಗಳು,1.6,0.0,1.40,0.0
ನಾನು ನಾಳೆ office ಗೆ ಹೋಗುತ್ತೇನೆ,1.2,0.0,1.40,0.0
ನನ್ನ phone number ೯೮೮೦೧೨೩೪೫೬,3.5,0.0,3.75,0.0
ಏನ್ ಮಾಡ್ತಿದೀಯ,3.0,0.0,3.00,0.0
ಕನ್‌ನಡ ಕನ್ನಡ,3.0,0.0,3.00,0.0


The segmentations results (especially for the character ಗೆ) show that when the Unigram vocabulary lacks the space-prefixed form of a piece, its most probable segmentation can be the space token (Ġ) followed by the bare piece. Whenever this happens, the word requires two tokens instead of one.

In [31]:
# Checking the number of single-space tokens in the unigram vocabulary
lone_u = counts["Unigram"].get("Ġ", 0) / sum(counts["Unigram"].values())
print(f"Lone-space tokens: {lone_u:.1%} of Unigram's Kannada tokens")

Lone-space tokens: 2.5% of Unigram's Kannada tokens


In [32]:
# Checking the number of single-space tokens in the BPE vocabulary
lone_b = counts["BPE"].get("Ġ", 0) / sum(counts["BPE"].values())
print(f"Lone-space tokens: {lone_b:.1%} of BPE's Kannada tokens")

Lone-space tokens: 0.0% of BPE's Kannada tokens


#### Checking the tokens that are only in Unigram but not in BPE and vice versa

In [33]:
only_unigram = u_new_tokens - b_new
only_bpe = b_new - u_new_tokens
print(f"Shared: {len(u_new_tokens & b_new)}, only Unigram: {len(only_unigram)}, only BPE: {len(only_bpe)}")

Shared: 2096, only Unigram: 1902, only BPE: 1902


In [34]:
def show(tokens, token_counts, k=40):
    for t in sorted(tokens, key=lambda t: -token_counts.get(t, 0))[:k]:
        print(f"{ec.to_text(t)!r:20} {token_counts.get(t, 0):>8,}")

In [35]:
print("\nOnly in the Unigram tokenizer (by Unigram usage):")
show(only_unigram, counts["Unigram"])


Only in the Unigram tokenizer (by Unigram usage):
'ಿದೆ'                  28,101
'ಿದ್ದಾರೆ'              27,420
'ಾಗ'                   23,096
'ಿರುವ'                 20,090
'ಿದ್ದ'                 18,635
'ುತ್ತದೆ'               17,803
'ಿಗೆ'                  15,125
'ಿತ'                   13,779
'ುವುದು'                12,983
'ಿದ್ದರು'               12,813
'ಂತ'                   12,200
'ಿಲ್ಲ'                 11,902
'ಿದ್ದು'                11,626
'ಿತ್ತು'                11,455
'ಿನ'                   10,752
'ಿಸುವ'                  8,861
'ಂಗ'                    8,657
'ಿತು'                   8,407
'ಿದರು'                  8,380
'ಂತೆ'                   8,251
'ಂಬ'                    8,141
'ಂಡ'                    8,074
'ಾಗಿ'                   8,054
'ುತ್ತಾರೆ'               7,969
'್ಯ'                    7,959
'ುತ್ತಿದೆ'               7,457
' ನಡೆಸ'                 7,235
'ಿಸಿದ'                  7,222
'್ಯಾ'                   7,186
'ಿಸಿ'                   6,759
'ುತ್ತಿರುವ'              6,645
'ುವಂತೆ'            

Many of these tokens are genuine suffixes and auxiliary verb forms (ಯಿಂದ, ಕೊಂಡು, ತ್ತದೆ, ದ್ದರು, ಕೊಳ್ಳ). The unigram tokenizer's suffix-boundary advantage leads to this.
<br><br>
Many characters are split incorrectly. Eg: ಸ�, �ರ್ಕಾರ, ಪ್ರ�, ನಿ�. These are byte fragments. 

In [36]:
print("\nOnly in the BPE tokenizer (by BPE usage):")
show(only_bpe, counts["BPE"])


Only in the BPE tokenizer (by BPE usage):
'ದು'                   40,582
'ಯು'                   22,518
'ಗ್'                   19,059
'ಸು'                   16,995
'ನೆ'                   16,766
' ರ'                   16,549
' ತ'                   15,589
'ದಾ'                   14,954
'ಯಿ'                   14,438
' ನ'                   14,420
'ಸಲು'                  13,755
'ಡು'                   13,660
'ದ್'                   13,279
'ದರೆ'                  12,844
'ಗು'                   12,645
'ದ್ದಾರೆ'               12,527
'ರದ'                   11,912
'ಸಿದ'                  11,597
' ವ'                   11,125
'ಲೆ'                   10,697
'ರೇ'                   10,528
' ಯ'                   10,342
' ಸ್'                  10,168
' ಕ್'                  10,132
' ದ'                    9,853
'ಕು'                    9,790
' ನಡೆ'                  9,523
' ಗ'                    9,299
' ಅಂ'                   8,948
'ಪು'                    8,339
'ದರು'                   8,331
' ಬಂ'                   8,2

#### Checking the Unigram tokenizer's fragment rate

In [37]:
for name, seg in segs.items():
    print(name, ec.fragment_rate(kn_texts, pre, seg))

BPE 0.002952622951418172
Unigram 0.0022260198618180753


The fragmentation rate on the Kannada evaluation set for the BPE tokenizer (with 4k new merges) is 0.00518922 (and for 2k new merges, it is 0.00947438).

#### Checking if the Unigram vocab has tokens beginning with split vowel signs

Also checking how many such tokens are present in both tokenizers' vocabs

In [38]:
def starts_with_mark(p):
    t = ec.to_text(p).lstrip(" ")
    return len(t) > 1 and unicodedata.category(t[0]).startswith("M")

In [39]:
for name, vocab, cnt in [("Unigram", u_new_tokens, counts["Unigram"]), ("BPE", b_new, counts["BPE"])]:
    
    marked = sorted((t for t in vocab if starts_with_mark(t)), key=lambda t: -cnt.get(t, 0))
    used = sum(cnt.get(t, 0) for t in marked) / sum(cnt.values())

    print(f"{name}: {len(marked)} mark-initial tokens, {used:.1%} of Kannada token usage")
    print("   ", [ec.to_text(t) for t in marked[:20]])


Unigram: 296 mark-initial tokens, 8.1% of Kannada token usage
    ['ಿದೆ', 'ಿದ್ದಾರೆ', 'ಾಗ', 'ಿರುವ', 'ಿದ್ದ', 'ುತ್ತದೆ', 'ಿಗೆ', 'ಿತ', 'ುವುದು', 'ಿದ್ದರು', 'ಂತ', 'ಿಲ್ಲ', 'ಿದ್ದು', 'ಿತ್ತು', 'ಿನ', 'ಿಸುವ', 'ಂಗ', 'ಿತು', 'ಿದರು', 'ಂತೆ']
BPE: 0 mark-initial tokens, 0.0% of Kannada token usage
    []
